# Why Most AIS Pings Result in No Prediction

This notebook traces every AIS ping through the prediction pipeline and measures
how many pings are lost at each filter stage, and why.

## Pipeline stages

```
All cargo pings
  └─ Slow-moving (SOG ≤ 1.5 kt)         ← port-candidate filter
       └─ Within port buffer (≤ 10 nm)  ← spatial filter
            └─ In-port pings            → NOT predicted (vessel is docked)
  └─ Sea pings (current_port is null)
       └─ Has origin port               ← needs at least one prior port visit
            └─ Has destination port     ← needs a future port visit
                 └─ Has voyage_id       → PREDICTED
```

## Data requirements

Run one of the two setup options below:

**Option A — cached data (fast):**  
The notebook reads `data/cache/labeled.parquet` and `data/cache/voyages.parquet`
produced by `notebooks/training.ipynb`.  Run that notebook first.

**Option B — raw AIS Parquet files:**  
Download AIS data with `data/ais/ais_fetcher.py` and place the resulting
`ais-2025-01-*.parquet` files in `data/ais/`.  The pipeline is then run here.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from shapely.geometry import Point

from src.methods import dms_to_dd
from src.voyage_creator import VoyageCreator

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Load data

In [ ]:
df_ports = pd.read_csv('../data/ports/ports.csv')
df_ports['lat_dd'] = df_ports['latitude'].apply(dms_to_dd)
df_ports['lon_dd'] = df_ports['longitude'].apply(dms_to_dd)
df_ports['geometry'] = df_ports.apply(lambda r: Point(r['lon_dd'], r['lat_dd']), axis=1)
gdf_ports = gpd.GeoDataFrame(df_ports, geometry='geometry', crs='EPSG:4326')
print(f'Loaded {len(gdf_ports)} ports')

In [ ]:
CACHE_DIR    = Path('../data/cache')
AIS_DIR      = Path('../data/ais')
labeled_path = CACHE_DIR / 'labeled.parquet'
voyages_path = CACHE_DIR / 'voyages.parquet'

if labeled_path.exists() and voyages_path.exists():
    # ── Option A: load from cache ──────────────────────────────────────────
    print('Loading labeled pings and voyages from cache ...')
    df_labeled = pq.read_table(labeled_path, use_pandas_metadata=False).to_pandas()
    df_labeled['base_date_time'] = pd.to_datetime(df_labeled['base_date_time'])

    df_voyages = pq.read_table(voyages_path, use_pandas_metadata=False).to_pandas()
    df_voyages['departure_time'] = pd.to_datetime(df_voyages['departure_time'])
    df_voyages['arrival_time']   = pd.to_datetime(df_voyages['arrival_time'])

    # Reconstruct raw AIS from labeled data (all columns present)
    df_cargo = df_labeled.copy()
    print('Loaded from cache.')

else:
    # ── Option B: run the full pipeline from raw Parquet files ─────────────
    parquet_files = sorted(AIS_DIR.glob('ais-2025-01-0*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(
            'No AIS Parquet files found in data/ais/.\n'
            'Run  python data/ais/ais_fetcher.py --start 2025-01-01 --end 2025-01-09\n'
            'or run notebooks/training.ipynb first to populate the cache.'
        )

    print(f'Reading {len(parquet_files)} Parquet file(s) ...')
    df_cargo = pd.concat(
        [pq.read_table(f, use_pandas_metadata=False).to_pandas() for f in parquet_files],
        ignore_index=True,
    )
    df_cargo['base_date_time'] = pd.to_datetime(df_cargo['base_date_time'])
    df_cargo = df_cargo.sort_values(['mmsi', 'base_date_time']).reset_index(drop=True)

    print('Running voyage pipeline (this may take a few minutes) ...')
    creator     = VoyageCreator(gdf_ports, radius_nm=10, max_speed_knots=1.5, gap_threshold_h=24)
    port_visits = creator.find_port_visits(df_cargo)
    df_labeled  = creator.label_pings(df_cargo, port_visits)
    df_labeled, df_voyages = creator.build_voyages(df_labeled, port_visits)

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    df_labeled.to_parquet(labeled_path, index=False)
    df_voyages.to_parquet(voyages_path, index=False)
    print('Pipeline complete and cached.')

print(f'\nTotal cargo pings : {len(df_labeled):,}')
print(f'Unique vessels    : {df_labeled["mmsi"].nunique():,}')
print(f'Voyages           : {len(df_voyages):,}')

## 2. Prediction funnel

Count pings at each stage of the pipeline and calculate the percentage that
survive to the next stage.

In [ ]:
total            = len(df_labeled)
n_in_port        = df_labeled['current_port'].notna().sum()
n_sea            = (df_labeled['current_port'].isna()).sum()
n_has_origin     = (df_labeled['current_port'].isna() & df_labeled['origin_port'].notna()).sum()
n_has_dest       = (df_labeled['current_port'].isna() & df_labeled['destination_port'].notna()).sum()
n_predicted      = (df_labeled['voyage_id'].notna() & df_labeled['destination_port'].notna()).sum()

funnel = pd.DataFrame({
    'Stage': [
        'All cargo pings',
        'In-port (SOG ≤ 1.5 kt & within 10 nm buffer)',
        'At sea (not in-port)',
        'At sea with known origin port',
        'At sea with known destination port',
        'Predicted (voyage_id + destination known)',
    ],
    'Ping count': [total, n_in_port, n_sea, n_has_origin, n_has_dest, n_predicted],
})
funnel['% of total'] = (funnel['Ping count'] / total * 100).round(1)
funnel['% of prev']  = funnel['Ping count'].pct_change().mul(100).round(1).fillna(0)

print(funnel.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']
bars    = ax.barh(funnel['Stage'][::-1], funnel['Ping count'][::-1], color=colors[::-1])

for bar, pct in zip(bars, funnel['% of total'][::-1]):
    ax.text(
        bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
        f'{pct:.1f}%', va='center', fontsize=9,
    )

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.set_xlabel('Number of pings')
ax.set_title('AIS ping prediction funnel')
plt.tight_layout()
plt.show()

## 3. In-port pings — vessels are docked, not en route

In-port pings are correctly excluded: a vessel sitting at a berth has no
meaningful remaining travel time.  This section characterises *which* ports
absorb the most pings.

In [ ]:
df_inport = df_labeled[df_labeled['current_port'].notna()].copy()

port_ping_counts = (
    df_inport.groupby('current_port')
    .size()
    .rename('ping_count')
    .sort_values(ascending=False)
)
print(f'In-port pings   : {len(df_inport):,}  ({len(df_inport)/total*100:.1f}% of all pings)')
print(f'Unique ports    : {port_ping_counts.index.nunique()}')
print(f'\nTop 15 ports by ping count:')
display(port_ping_counts.head(15).to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# SOG distribution of in-port pings
sog_inport = df_inport['sog'].dropna().clip(upper=5)
axes[0].hist(sog_inport, bins=30, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].axvline(1.5, color='red', linestyle='--', label='Max speed threshold (1.5 kt)')
axes[0].set_xlabel('SOG (knots, capped at 5)')
axes[0].set_ylabel('Ping count')
axes[0].set_title('SOG distribution — in-port pings')
axes[0].legend()

# Top 15 ports
top15 = port_ping_counts.head(15)
axes[1].barh(top15.index[::-1], top15.values[::-1], color='steelblue')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
axes[1].set_xlabel('Ping count')
axes[1].set_title('Top 15 ports by in-port pings')

plt.tight_layout()
plt.show()

## 4. Sea pings without voyage context

A sea ping receives no prediction if it lacks either an **origin port** (no
preceding port visit detected in the buffer) or a **destination port** (no
future port visit detected).  These are the three sub-cases:

| Sub-case | Origin | Destination | Explanation |
|----------|--------|-------------|-------------|
| Orphan — no context | NaN | NaN | Vessel appeared mid-ocean without a tracked port visit on either side |
| Origin only | known | NaN | Vessel left port but has not yet arrived at the next port (still being observed) |
| Destination only | NaN | known | Vessel is heading towards a port for the first time in the dataset window |

In [ ]:
df_sea = df_labeled[df_labeled['current_port'].isna()].copy()

has_origin = df_sea['origin_port'].notna()
has_dest   = df_sea['destination_port'].notna()
has_voyage = df_sea['voyage_id'].notna()

sea_breakdown = pd.Series({
    'No origin, no destination (orphan)': (~has_origin & ~has_dest).sum(),
    'Origin only (no future port yet)':   (has_origin & ~has_dest).sum(),
    'Destination only (no prior port)':   (~has_origin & has_dest).sum(),
    'Both origin + destination (predictable)': (has_origin & has_dest).sum(),
}, name='ping_count')

sea_breakdown_pct = (sea_breakdown / len(df_sea) * 100).round(1).rename('% of sea pings')
print(f'Sea pings total: {len(df_sea):,}\n')
display(pd.concat([sea_breakdown, sea_breakdown_pct], axis=1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
palette = ['#C44E52', '#DD8452', '#937860', '#55A868']
bars = ax.barh(sea_breakdown.index[::-1], sea_breakdown.values[::-1], color=palette[::-1])

for bar, pct in zip(bars, sea_breakdown_pct.values[::-1]):
    ax.text(
        bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
        f'{pct:.1f}%', va='center', fontsize=9,
    )

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.set_xlabel('Number of sea pings')
ax.set_title('Sea pings broken down by voyage context')
plt.tight_layout()
plt.show()

## 5. Vessel-level coverage

How many vessels complete at least one full voyage (origin → destination) vs.
those that are only ever seen in a single state?

In [ ]:
vessel_stats = (
    df_labeled
    .groupby('mmsi')
    .agg(
        total_pings         = ('mmsi', 'count'),
        in_port_pings       = ('current_port', lambda x: x.notna().sum()),
        has_origin          = ('origin_port', lambda x: x.notna().any()),
        has_destination     = ('destination_port', lambda x: x.notna().any()),
        predicted_pings     = ('voyage_id', lambda x: x.notna().sum()),
        n_voyages           = ('voyage_id', lambda x: x.dropna().nunique()),
    )
    .reset_index()
)

vessel_stats['pct_predicted'] = (
    vessel_stats['predicted_pings'] / vessel_stats['total_pings'] * 100
).round(1)

# Classify each vessel
def classify(row):
    if row['n_voyages'] > 0:
        return 'Has ≥1 complete voyage'
    elif row['in_port_pings'] > 0 and row['predicted_pings'] == 0:
        return 'Port-only (never left observed ports)'
    elif not row['has_origin'] and not row['has_destination']:
        return 'Orphan (no port context at all)'
    else:
        return 'Incomplete voyage (missing one end)'

vessel_stats['category'] = vessel_stats.apply(classify, axis=1)

cat_counts = vessel_stats['category'].value_counts()
print('Vessel categories:')
for cat, n in cat_counts.items():
    print(f'  {cat:<45} {n:>5} vessels  ({n/len(vessel_stats)*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Pie chart of vessel categories
palette_cat = ['#55A868', '#4C72B0', '#DD8452', '#C44E52']
cat_counts_sorted = cat_counts.sort_values(ascending=False)
axes[0].pie(
    cat_counts_sorted.values,
    labels=cat_counts_sorted.index,
    autopct='%1.1f%%',
    colors=palette_cat[:len(cat_counts_sorted)],
    startangle=90,
    textprops={'fontsize': 9},
)
axes[0].set_title('Vessel breakdown by prediction eligibility')

# Distribution of voyage counts per vessel
voyage_hist = vessel_stats['n_voyages'].value_counts().sort_index().head(15)
axes[1].bar(voyage_hist.index.astype(str), voyage_hist.values, color='steelblue')
axes[1].set_xlabel('Number of completed voyages per vessel')
axes[1].set_ylabel('Number of vessels')
axes[1].set_title('Distribution of completed voyages per vessel')

plt.tight_layout()
plt.show()

In [ ]:
print('Predicted-ping coverage per vessel category (% of each vessel\'s pings that are predicted):')
display(
    vessel_stats.groupby('category')['pct_predicted']
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(1)
)

## 6. Temporal pattern — when do predictions fail?

Hourly ping rates by category: predictions may be absent at the *start* of the
dataset window (before any port visit has been observed) and at the *end* (before
the next port visit is confirmed).

In [ ]:
df_labeled['_hour'] = df_labeled['base_date_time'].dt.floor('h')

df_labeled['_ping_category'] = 'Sea — no prediction'
df_labeled.loc[df_labeled['current_port'].notna(), '_ping_category'] = 'In-port'
df_labeled.loc[
    df_labeled['voyage_id'].notna() & df_labeled['destination_port'].notna(),
    '_ping_category'
] = 'Predicted'

hourly = (
    df_labeled
    .groupby(['_hour', '_ping_category'])
    .size()
    .unstack(fill_value=0)
)

# Ensure all three categories exist
for col in ['In-port', 'Sea — no prediction', 'Predicted']:
    if col not in hourly.columns:
        hourly[col] = 0

fig, ax = plt.subplots(figsize=(13, 4))
hourly[['In-port', 'Sea — no prediction', 'Predicted']].plot.area(
    ax=ax,
    color=['#4C72B0', '#C44E52', '#55A868'],
    alpha=0.8,
    linewidth=0,
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax.set_xlabel('Time')
ax.set_ylabel('Pings per hour')
ax.set_title('Hourly AIS pings by prediction outcome')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 7. Voyage gap analysis

Two specific boundary effects at the edges of the data window inflate the
"no prediction" count:

1. **Warm-up period** — vessels seen at sea at the very start of the window
   have no prior port visit recorded, so `origin_port` is NaN.
2. **Cool-down period** — vessels that never arrive at their next port within
   the window have no future visit recorded, so `destination_port` is NaN.

In [ ]:
if len(df_voyages) > 0:
    data_start = df_labeled['base_date_time'].min()
    data_end   = df_labeled['base_date_time'].max()

    # Voyage duration distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(
        df_voyages['duration_hours'].clip(upper=200),
        bins=40, color='steelblue', edgecolor='white', linewidth=0.4,
    )
    axes[0].set_xlabel('Voyage duration (hours, capped at 200)')
    axes[0].set_ylabel('Number of voyages')
    axes[0].set_title('Distribution of voyage durations')

    # Ping count per voyage
    axes[1].hist(
        df_voyages['ping_count'].clip(upper=5000),
        bins=40, color='steelblue', edgecolor='white', linewidth=0.4,
    )
    axes[1].set_xlabel('Sea pings per voyage (capped at 5 000)')
    axes[1].set_ylabel('Number of voyages')
    axes[1].set_title('Distribution of pings per voyage')

    plt.tight_layout()
    plt.show()

    # How many voyages touch the data window edges?
    edge_tol = pd.Timedelta(hours=2)
    starts_at_edge = (df_voyages['departure_time'] - data_start).abs() < edge_tol
    ends_at_edge   = (df_voyages['arrival_time']   - data_end).abs()   < edge_tol
    print(f'Voyages that start within 2 h of data window start : {starts_at_edge.sum()}')
    print(f'Voyages that end   within 2 h of data window end   : {ends_at_edge.sum()}')
    print(f'Total voyages                                       : {len(df_voyages)}')
else:
    print('No voyages found — cannot plot voyage gap analysis.')

## 8. SOG threshold sensitivity

The port-candidate filter uses `SOG ≤ 1.5 kt`.  A higher threshold would admit
more in-port pings but also more slowly-moving sea pings, which could cause
false port detections.  This section shows the SOG distribution of sea pings
that sit just above the current threshold.

In [ ]:
MAX_SPEED = 1.5  # current threshold

sog_sea = df_labeled.loc[df_labeled['current_port'].isna(), 'sog'].dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sog_sea.clip(upper=25), bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(MAX_SPEED, color='red', linestyle='--', linewidth=1.5,
           label=f'Current SOG threshold ({MAX_SPEED} kt)')

# Annotate pings within 0–3 kt
n_slow_sea = (sog_sea <= 3).sum()
ax.axvspan(0, 3, alpha=0.12, color='orange', label=f'SOG ≤ 3 kt sea pings: {n_slow_sea:,}')

ax.set_xlabel('SOG (knots, capped at 25)')
ax.set_ylabel('Ping count')
ax.set_title('SOG distribution of sea pings')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax.legend()
plt.tight_layout()
plt.show()

# What fraction of sea pings are very slow (potential anchoring / waiting)?
brackets = [(0, 0.5), (0.5, 1.5), (1.5, 3), (3, 6), (6, 25)]
print('SOG distribution of sea pings:')
for lo, hi in brackets:
    mask = (sog_sea >= lo) & (sog_sea < hi)
    print(f'  [{lo:4.1f}, {hi:4.1f}) kt : {mask.sum():>8,}  ({mask.mean()*100:5.1f}%)')

## 9. Port coverage — are the right ports in the reference set?

The 657 US ports in `data/ports/ports.csv` may not cover every anchorage or
terminal where cargo vessels slow down.  If a vessel idles within a port's
physical waters but the port is not in the reference set, the visit is missed
and the voyage cannot be reconstructed.

In [ ]:
# Ports that appear in voyages vs. the full port list
if len(df_voyages) > 0:
    ports_in_voyages = set(df_voyages['departure_port']) | set(df_voyages['arrival_port'])
    all_port_names   = set(gdf_ports['portName'])
    active_ports     = ports_in_voyages & all_port_names
    unused_ports     = all_port_names - ports_in_voyages

    print(f'Total ports in reference set : {len(all_port_names)}')
    print(f'Ports observed in voyages    : {len(active_ports)}  ({len(active_ports)/len(all_port_names)*100:.1f}%)')
    print(f'Ports never visited          : {len(unused_ports)}  ({len(unused_ports)/len(all_port_names)*100:.1f}%)')

    # How many voyages per active port?
    dep_counts = df_voyages['departure_port'].value_counts()
    arr_counts = df_voyages['arrival_port'].value_counts()
    port_voyage_counts = (dep_counts.add(arr_counts, fill_value=0)).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 4))
    top20 = port_voyage_counts.head(20)
    ax.bar(range(len(top20)), top20.values, color='steelblue')
    ax.set_xticks(range(len(top20)))
    ax.set_xticklabels(top20.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Voyage departures + arrivals')
    ax.set_title('Top 20 most active ports (by voyage count)')
    plt.tight_layout()
    plt.show()
else:
    print('No voyages found.')

## 10. Slow sea pings — anchoring or missing port?

Sea pings with very low SOG (e.g. < 1.5 kt) that are *not* near any known
port likely represent vessels at anchor, waiting for a berth, or transiting
a channel.  These pings are correctly excluded from predictions but could
indicate missing ports in the reference set.

In [ ]:
# Sea pings that are slow but not matched to any port
df_slow_sea = df_labeled[
    df_labeled['current_port'].isna() &
    (df_labeled['sog'] <= 1.5)
].copy()

print(f'Slow sea pings (SOG ≤ 1.5, not in-port): {len(df_slow_sea):,}')
print(f'Unique vessels                           : {df_slow_sea["mmsi"].nunique():,}')

# Voyage context of slow sea pings
has_v = df_slow_sea['voyage_id'].notna()
has_d = df_slow_sea['destination_port'].notna()
print(f'\nOf these slow sea pings:')
print(f'  With voyage context  : {(has_v & has_d).sum():,}')
print(f'  Without context      : {(~has_v | ~has_d).sum():,}')

if len(df_slow_sea) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(
        df_slow_sea['longitude'].clip(-130, -60),
        df_slow_sea['latitude'].clip(20, 55),
        s=1, alpha=0.15, color='#C44E52', rasterized=True,
    )
    ax.set_xlim(-130, -60)
    ax.set_ylim(20, 55)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Slow sea pings (SOG ≤ 1.5 kt, not matched to any port)\nPossible anchoring or unregistered port activity')
    plt.tight_layout()
    plt.show()

## 11. Summary of root causes

The table below consolidates the ping counts into labelled root causes.

In [ ]:
df_sea = df_labeled[df_labeled['current_port'].isna()]
has_origin = df_sea['origin_port'].notna()
has_dest   = df_sea['destination_port'].notna()

causes = pd.DataFrame([
    {
        'Root cause': 'Vessel is in port (SOG ≤ 1.5 kt, within 10 nm)',
        'Why no prediction': 'Vessel is docked — no travel time to predict',
        'Ping count': int(df_labeled['current_port'].notna().sum()),
    },
    {
        'Root cause': 'Orphan sea ping (no origin, no destination)',
        'Why no prediction': 'Vessel entered dataset mid-ocean, never near any port',
        'Ping count': int((~has_origin & ~has_dest).sum()),
    },
    {
        'Root cause': 'Sea ping — origin only (no future port in window)',
        'Why no prediction': 'Data window ends before vessel reaches next port',
        'Ping count': int((has_origin & ~has_dest).sum()),
    },
    {
        'Root cause': 'Sea ping — destination only (no prior port in window)',
        'Why no prediction': 'Data window starts after vessel left its prior port',
        'Ping count': int((~has_origin & has_dest).sum()),
    },
    {
        'Root cause': 'Predicted — both origin + destination known',
        'Why no prediction': '—',
        'Ping count': int(n_predicted),
    },
])

causes['% of all pings'] = (causes['Ping count'] / total * 100).round(1)
causes = causes.sort_values('Ping count', ascending=False).reset_index(drop=True)

print(f'Total pings: {total:,}\n')
display(causes)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors_summary = ['#4C72B0', '#C44E52', '#DD8452', '#937860', '#55A868']
wedges, texts, autotexts = ax.pie(
    causes['Ping count'],
    labels=None,
    autopct=lambda p: f'{p:.1f}%' if p > 2 else '',
    colors=colors_summary[:len(causes)],
    startangle=90,
    pctdistance=0.75,
)
ax.legend(
    wedges, causes['Root cause'],
    loc='lower center', bbox_to_anchor=(0.5, -0.22),
    fontsize=8, ncol=1,
)
ax.set_title('AIS ping classification by prediction outcome')
plt.tight_layout()
plt.show()

## 12. Recommendations

Based on the analysis above, the main levers for increasing prediction coverage are:

| Lever | Impact | Complexity |
|-------|--------|------------|
| **Extend the data window** (more days before / after) | Reduces warm-up and cool-down gaps — the largest single source of missing context | Low |
| **Expand the port reference set** (add anchorages, terminals) | Captures slow sea pings that are actually at unregistered ports | Medium |
| **Increase buffer window in the streaming predictor** (`buffer_hours`) | Retains more history per vessel so earlier port visits are still visible | Low |
| **Widen the port buffer radius** (`voyage_radius_nm`) | Catches vessels that slow down slightly outside the 10 nm zone | Medium — higher false-positive rate |
| **Predict for single-ended voyages** | Generate predictions even when only origin or only destination is known (with higher uncertainty) | High — requires retraining |
| **Pre-seed vessel history** from a longer historical archive | Eliminates the warm-up period entirely for known vessels | Medium |